In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

ground_truth

[{'question': "Can I join the course now that it's already running?",
  'document': '74eb249bbf'},
 {'question': 'How do I enroll if I discovered the course just now?',
  'document': '74eb249bbf'},
 {'question': 'Will I be able to get a certificate if I sign up after the start date?',
  'document': '74eb249bbf'},
 {'question': 'Do I need to finish and submit my project to receive a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What’s the deadline for project submission if I join late?',
  'document': '74eb249bbf'},
 {'question': 'When will I get a confirmation email after registering for the LLM Zoomcamp?',
  'document': '977bf7786c'},
 {'question': 'Do I need to wait for a confirmation email before starting the course?',
  'document': '977bf7786c'},
 {'question': 'Can I start learning and submitting homework without a confirmation email?',
  'document': '977bf7786c'},
 {'question': 'Is my registration verified against a list before the course begins?',
  'document': '977b

In [2]:
from pathlib import Path
import sys
import os

# 1. Get the absolute path of the directory containing this script
parent_dir = Path(os.getcwd()).resolve().parent

# 3. Add the parent directory to Python's search path
sys.path.append(str(parent_dir))

##############################################################################
# 4. Import the function normally
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [3]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [4]:
# Start with one ground truth record for testing
q = ground_truth[0]
doc_id = q["document"]
results = text_search(query=q["question"])

for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

74eb249bbf == 74eb249bbf: True
977bf7786c == 74eb249bbf: False
a8a7fef016 == 74eb249bbf: False
04919992b3 == 74eb249bbf: False
69d122f12e == 74eb249bbf: False


[1, 0, 0, 0, 0]

In [5]:
# Put this logic into a function
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [6]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [7]:
# Do this for the first 15 ground truth questions:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

relevance_total_text

  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0]]

In [8]:
# now, we makes the compute_relevant_text generic,
# with ability to pass in another mode of search, e.g. vector search, or hybrid search
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

# Likewise, the total relevance function too:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [9]:
# Try running this for all ground truth questions, using sparse search:
relevance_total = compute_relevance_total(ground_truth, text_search)
relevance_total

  0%|          | 0/565 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0,

# Next up, we evaluate this results of 'Search'

In [10]:
# Hit Rate and Recall
def hit_rate(relevance):
    # This count the number of lines in which at least one of the top 5 results is the correct document (i.e., has a 1 in it)
    # And returns the fraction of lines that have at least one correct document in the top 5 results
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

hit_rate(relevance_total)


0.7823008849557522

In [11]:
# Mean Reciprocal Rank (MRR)
# This weights in the ranking, giving higher scores to correct documents that appear higher in the ranking. 
# The reciprocal rank is 1/rank, where rank is the position of the first relevant document in the list of results.
# pos 1: 1.0
# pos 2: 0.5
# pos 3: 0.333
# etc. 
# not found: 0.0

def mrr(relevance):
    # Hit rate is the upper bound of MRR
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

mrr(relevance_total)

0.6502949852507371

# Wrapping the metrics in a reusable evaluation function:

In [12]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/565 [00:00<?, ?it/s]

{'hit_rate': 0.7823008849557522, 'mrr': 0.6502949852507371}

# Search parameter Tuning

In [13]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [14]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(
                f"Evaluating question_boost={question_boost},"
                f" answer_boost={answer_boost},"
                f" section_boost={section_boost}..."
            )
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/565 [00:00<?, ?it/s]

,question,answer,section,hit_rate,mrr
7,1.0,4.0,0.2,0.943363,0.832035
6,1.0,4.0,0.1,0.946903,0.830767
23,2.0,10.0,0.5,0.950442,0.827168
22,2.0,10.0,0.2,0.950442,0.824897
8,1.0,4.0,0.5,0.936283,0.824307
4,1.0,2.0,0.2,0.945133,0.824189
19,2.0,4.0,0.2,0.946903,0.823776
3,1.0,2.0,0.1,0.946903,0.823776
35,5.0,10.0,0.5,0.946903,0.823776
21,2.0,10.0,0.1,0.950442,0.823717
